# Suite CCS — C and C++ simplified by ISO/IEC 9899 and ISO/IEC 14882, under their holder's ABI

`C.Simplifiers.OfAny(term, scope)` and `Cpp.Simplifiers.OfAny(term, scope)` simplify a term of the C family by its
language's standard, every kind the evaluation inventory gives a value folding: what the scope decides folds into the
literal of its value, as the language writes it (a suffix by type, `nullptr`, a floating value a binary64 literal spells
exactly), a value of a type no literal has into a cast of a literal (`(short)3`), and, in C, an array into a compound
literal (`(int[]){1, 2}`), each a closed constructor expression; the rest stays, the residual, its subterms simplified. The types it computes in are the stored domains its
holder reads, its own dialect's and its ABI's (`abi`, x86-64's System V ABI unless a case names AAPCS64's), so `sizeof`
and `_Generic` read a type's layout and identity there. What has no rule stays too, named with its class in the
evaluation inventory (`residues`): the object model and declarations (`d`), an operand's declared type (`c`), a layout
no ABI dialect stores (`a`), an initializer alone and a type name (`b`). What the standard makes an error still raises.
Terms are written through the dialects' writers and builders, in a transaction (`term`); `text` gives a residual's
source, as the printer writes it, and `rule` the folding rules' value of a term whose value has no literal.

In [ ]:
import math

from mbse.Expressions import Framework
from mbse.Expressions.Arm import AAPCS64
from mbse.Expressions.C import Expressions as CDialect, Preprocessor as CPreprocessor, Types as CTypes
from mbse.Expressions.C.Expressions import Expressions as C, Simplifiers as CV, Text as CText
from mbse.Expressions.Cpp import Expressions as CppDialect, RegExp as CppRegExp, Types as CppTypes
from mbse.Expressions.Cpp.Expressions import Expressions as X, Simplifiers as XV, Text as XText
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.POSIX import BRE as PosixBRE, ERE as PosixERE
from mbse.Expressions.SystemV import AMD64
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.Framework import Terms as F
from mbse.Expressions.Framework.Simplifiers import NotEvaluated
from mbse.Schemas.Framework import Builders
from support import raises

store = Builders.Store()
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install, C.install,
                X.install, AMD64.install, AAPCS64.install, CTypes.install, CppTypes.install, CPreprocessor.install,
                CppRegExp.install, PosixBRE.install, PosixERE.install]:
    store.transaction(install)
for define in [C.define, X.define, AMD64.define, AAPCS64.define]:  # the types C, C++ and each ABI define
    store.transaction(define)


def amd64():
    """The System V AMD64 ABI, read within the transaction open: the default ABI the cases read their terms under."""
    return store.Instances.Expressions.SystemV.AMD64


def aapcs64():
    """Arm's AAPCS64 ABI, read within the transaction open."""
    return store.Instances.Expressions.Arm.AAPCS64


def term(spec, of=C, abi=None):
    """The term `spec` writes in the dialect `of` (C or C++), a native value as its literal, the root of a holder built
    within the transaction open that names the ABI its terms are read under (none: AMD64's); the term, a visitor of
    that transaction, lives within it."""
    abi = amd64() if abi is None else abi
    return F.term(store.Expressions.Framework.Expression(lambda e: e.root(of.DIALECT.spec(spec)).abi([abi])))


def simplified(spec, scope=None, of=C, abi=None):
    """The simplification of the term `spec` writes, in `scope`."""
    return (CV if of is C else XV).OfAny(term(spec, of, abi), scope)


def text(spec, scope=None, of=C, abi=None):
    """The source of the residual of the term `spec` writes, simplified in `scope`."""
    residual = simplified(spec, scope, of, abi).residual
    abi = amd64() if abi is None else abi
    holder = store.Expressions.Framework.Expression(lambda e: e.root(residual).abi([abi]))
    return (CText if of is C else XText).ToText(holder)


def value(spec, scope=None, of=C, abi=None):
    """The value the term `spec` writes simplifies to, in `scope`: its literal root's."""
    return simplified(spec, scope, of, abi).evaluate()


def rule(spec, scope=None, of=C, abi=None):
    """The folding rules' value of the term `spec` writes, a `Typed`: where no literal spells it."""
    rules = CV if of is C else XV
    return rules.RULES.evaluate(term(spec, of, abi), scope if isinstance(scope, rules.Scope) else rules.Scope(scope or {}))


def reasons(spec, scope=None, of=C, abi=None):
    """Why what stayed in the residual stayed: each residue's reason and class."""
    return [(residue.reason, residue.category) for residue in simplified(spec, scope, of, abi).residues]


# The builders of the terms the cases write: a type name, of its keywords or a type's name, and an abstract declarator
def keywords(*words, declarator=None):
    specifiers = [lambda s, w=w: s.primitivetypespecifier(lambda p: p.keyword(w)) if w not in ("const", "volatile")
                  else s.cvqualifier(lambda q: q.keyword(w)) for w in words]
    return lambda u: u.typeid(lambda i: i.specifiers(specifiers) if declarator is None else i.specifiers(specifiers)
                              .declarator(declarator))


def named(name):
    return lambda u: u.typeid(lambda i: i.specifiers([lambda s: s.namedtypespecifier(
        lambda n: n.name(lambda m: m.identifier(lambda d: d.spelling(name))))]))


def bits(width, unsigned=False):
    """`_BitInt(width)`, or `unsigned _BitInt(width)`."""
    sign = [lambda s: s.primitivetypespecifier(lambda p: p.keyword("unsigned"))] if unsigned else []
    return lambda u: u.typeid(lambda i: i.specifiers([*sign, lambda s: s.bitintspecifier(lambda b: b.width(C.literal(width)))]))


def array(size=None, inner=None):
    """An array declarator, of `size` (none: its initializer's), around `inner`."""
    def declarator(d):
        def a(r):
            r = r if size is None else r.size(C.DIALECT.spec(size))
            return r if inner is None else r.declarator(inner)
        return d.arraydeclarator(a)
    return declarator


pointer = lambda d: d.pointerdeclarator(lambda p: p)  # noqa: E731
x, y = C.identifier("x"), C.identifier("y")
nullptr = lambda u: u.constant(lambda r: r.singleton("nullptr"))  # noqa: E731


def paren(spec, of=C):
    return lambda u: u.parenthesizedexpression(lambda r: r.expression(of.DIALECT.spec(spec)))


def extension(spec):
    return lambda u: u.extensionexpression(lambda r: r.operand(C.DIALECT.spec(spec)))


def string(value, prefix=None):
    return lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(value)) if prefix is None else
                               r.value(lambda v: v.as_scalar(value)).prefix(prefix))


def concat(*parts):
    return lambda u: u.concatenatedstring(lambda r: r.parts([C.DIALECT.spec(part) for part in parts]))


def gnu(condition, alternative):
    """GNU's `condition ?: alternative`."""
    return lambda u: u.conditionalexpression(lambda r: r.condition(C.DIALECT.spec(condition))
                                             .alternative(C.DIALECT.spec(alternative)))


def sizeof(operand, of=C):
    return lambda u: u.sizeofexpression(lambda r: r.operand(of.DIALECT.spec(operand)))


def alignof(operand):
    return lambda u: u.alignofexpression(lambda r: r.keyword("_Alignof").operand(C.DIALECT.spec(operand)))


def generic(controlling, *associations):
    """`_Generic(controlling, type: value, ..., default: value)`, `default`'s type None."""
    return lambda u: u.genericselection(lambda r: r.controlling(C.DIALECT.spec(controlling)).associations([
        (lambda a, t=t, v=v: a.value(C.DIALECT.spec(v)) if t is None else a.type(t).value(C.DIALECT.spec(v)))
        for t, v in associations]))


def compound(type_, *items):
    """`(type){items}`."""
    return lambda u: u.compoundliteralexpression(lambda r: r.type(type_).initializer(
        lambda i: i.items([C.DIALECT.spec(item) for item in items])))


def designated(index, item, last=None):
    """`[index] = item`, or GNU's `[index ... last] = item`."""
    return lambda u: u.designatedinitializer(lambda r: r.designators([lambda d: d.indexdesignator(
        lambda i: i.index(C.literal(index)) if last is None else i.index(C.literal(index)).last(C.literal(last)))])
        .initializer(lambda i: i.equalinitializer(lambda e: e.value(C.DIALECT.spec(item)))))


# C++'s
def ident(name):
    return X.identifier(name)


def qualified(*names):
    return lambda u: u.idexpression(lambda r: r.name(lambda n: n.qualifiedname(lambda q: q.qualifiers([
        lambda m, s=s: m.identifier(lambda i: i.spelling(s)) for s in names[:-1]]).name(
        lambda m: m.identifier(lambda i: i.spelling(names[-1]))))))


this_ = lambda u: u.thisexpression(lambda r: r)  # noqa: E731


def udl(value_, suffix):
    return lambda u: u.userdefinedliteral(lambda r: r.literal(lambda l: l.value(lambda v: v.as_scalar(value_))).suffix(suffix))


def functional(keyword, *arguments, braced=False):
    """`keyword(arguments)`, or `keyword{arguments}`, a functional cast; a keyword C++ does not write names a type."""
    specs = [X.DIALECT.spec(argument) for argument in arguments]
    specifier = (lambda s: s.primitivetypespecifier(lambda p: p.keyword(keyword))) if keyword in (
        "int", "unsigned", "double", "bool", "char", "void") else (lambda s: s.namedtypespecifier(
            lambda n: n.name(lambda m: m.identifier(lambda i: i.spelling(keyword)))))
    initializer = (lambda i: i.initializerlist(lambda l: l.items(specs))) if braced else (
        lambda i: i.parenthesizedinitializer(lambda p: p.arguments(specs)))
    return lambda u: u.functionalcastexpression(lambda r: r.type(specifier).initializer(initializer))


def static_cast(type_, operand, operator="static_cast"):
    return lambda u: u.namedcastexpression(lambda r: r.operator(operator).type(type_).operand(X.DIALECT.spec(operand)))


def sizeof_pack(name):
    return lambda u: u.sizeofpackexpression(lambda r: r.pack(lambda i: i.spelling(name)))


def expansion(pattern):
    return lambda u: u.packexpansion(lambda r: r.pattern(X.DIALECT.spec(pattern)))


def fold(left, operator, right):
    """`(left op ... op right)`; a unary fold's other operand None."""
    def record(r):
        r = r.operator(operator)
        r = r if left is None else r.left(X.DIALECT.spec(left))
        return r if right is None else r.right(X.DIALECT.spec(right))
    return lambda u: u.foldexpression(record)


def throw_(operand=None):
    return lambda u: u.throwexpression(lambda r: r if operand is None else r.operand(X.DIALECT.spec(operand)))

<a id="CCS-01"></a>
## CCS-01 · Literals written back as each language writes them: a suffix by type, a typedef's value as the type it is, a floating value a binary64 literal spells; a value of a type with no literal a cast of a literal, its promoted arithmetic folding

In [ ]:
with store.transaction():
    # A value folds into the literal of its type, its suffix as C writes it; a typedef's value as the standard type it is
    for spec, written in [(C.binary("+", C.literal(1, "u"), 1), "2u"), (C.binary("*", C.literal(2, "l"), 3), "6l"),
                          (C.binary("+", C.literal(1, "ull"), 1), "2ull"), (C.binary("<", 1, 2), "true"),
                          (C.binary("+", C.cast(named("int64_t"), 5), 1), "6l"), (C.cast(named("size_t"), 7), "7ul"),
                          (C.binary("*", C.literal(1.5, "f"), 2), "3.0f"), (C.binary("*", C.literal(1.5, "L"), 2), "3.0L"),
                          (C.binary("+", C.literal(0.1, "dd"), C.literal(0.2, "dd")), "0.3dd"),
                          (C.binary("*", C.literal(0.5, "f16"), 3), "1.5f16"), (string("hi"), '"hi"'), (nullptr, "nullptr")]:
        assert text(spec) == written and simplified(spec).literal, (written, text(spec))
    assert value(C.binary("+", C.literal(1, "u"), 1)) == 2 and value(C.binary("*", C.literal(1.5, "L"), 2)) == "3"  # natives
    # A floating value no binary64 literal spells exactly stays, as does a non-finite one; its value is the rules'
    for abi, held in [(amd64(), "0.33333333333333333334"), (aapcs64(), "0.3333333333333333333333333333333333")]:
        assert text(C.binary("/", C.literal(1.0, "L"), 3), abi=abi) == "1.0L / 3" and rule(C.binary("/", C.literal(1.0, "L"), 3), abi=abi).value == held
    assert text(C.binary("/", 1.0, 0.0)) == "1.0 / 0.0" and rule(C.binary("/", 1.0, 0.0)).value == math.inf
    # A value of a type with no literal is a cast of a literal that converts to it, a closed constructor expression: a char,
    # a short, a uint8_t's as the unsigned char it is; each ABI's char its own
    assert text(C.cast(keywords("char"), C.binary("+", 100, 100))) == "(char)-56" and text(C.cast(keywords("char"), 200), abi=aapcs64()) == "(char)200"
    assert simplified(C.cast(keywords("char"), 200)).constructed and not simplified(C.cast(keywords("char"), 200)).literal
    assert value(C.cast(keywords("char"), 200)) == -56 and rule(C.cast(keywords("char"), 200)).value == -56
    assert text(C.cast(keywords("short"), C.binary("+", 1, 2))) == "(short)3" and text(C.cast(named("uint8_t"), 300)) == "(unsigned char)44"
    assert text(C.cast(keywords("unsigned", "short"), 7)) == "(unsigned short)7" and text(C.cast(named("int8_t"), 200)) == "(signed char)-56"
    assert text(C.binary("+", x, C.cast(keywords("char"), 200))) == "x + (char)-56"  # a subterm, where the root stays
    # Its arithmetic, promoted to int, folds, as each ABI says: char is signed under AMD64, unsigned under AAPCS64
    assert text(C.binary("+", C.cast(keywords("char"), 200), 0)) == "-56" and text(C.binary("+", C.cast(keywords("char"), 200), 0), abi=aapcs64()) == "200"
    # A typedef of an integer type is that type in C; C++'s character types are types of their own
    assert text(C.cast(named("wchar_t"), 65)) == "65" and text(C.cast(named("wchar_t"), 65), abi=aapcs64()) == "65u"
    assert text(X.cast(named("wchar_t"), 65), of=X) == "(wchar_t)65" and text(X.cast(named("int64_t"), 65), of=X) == "65l"
    assert text(X.cast(named("char16_t"), 65), of=X) == "(char16_t)65" and text(X.cast(keywords("char"), 65), of=X) == "(char)65"
    # C++ writes its own extended floating types' suffixes
    assert text(X.binary("*", X.literal(1.5, "f16"), 2), of=X) == "3.0f16" and text(X.binary("*", X.literal(1.5, "bf16"), 2), of=X) == "3.0bf16"

<a id="CCS-02"></a>
## CCS-02 · Parentheses, `__extension__`, adjacent strings, the comma and a cast to `void`

In [ ]:
with store.transaction():
    # Parentheses and GNU's `__extension__` are their operand; where it stays, the printer keeps what precedence needs
    assert text(paren(C.binary("+", 1, 2))) == "3" and text(C.binary("*", paren(C.binary("+", x, 2)), 3)) == "(x + 2) * 3"
    assert text(extension(C.binary("+", 1, 2))) == "3" and text(paren(x), {"x": 4}) == "4"
    # Adjacent string literals are one string; a prefixed one stays, as a prefixed string literal does
    assert text(concat("ab", "cd")) == '"abcd"' and value(concat("ab", string("cd"))) == "abcd"
    assert reasons(concat("a", string("b", "L"))) == [("C literal of wchar_t string is not simplified", "a")]
    # The comma: its left operand evaluated, then its right's value; a known left operand leaves the right
    assert text(C.binary(",", 1, 2)) == "2" and text(C.binary(",", 1, x)) == "x" and text(C.binary(",", x, 1)) == "x, 1"
    with raises(ZeroDivisionError, match="integer division by zero"):
        value(C.binary(",", C.binary("/", 1, 0), 2))
    # A cast to void evaluates its operand, to no value: it stays, its operand simplified
    assert text(C.cast(keywords("void"), C.binary("+", 1, 2))) == "(void)3" and rule(C.cast(keywords("void"), 1)).value is CV.VOID
    assert text(C.binary(",", C.cast(keywords("void"), 1), 2)) == "2"
    with raises(TypeError, match="invalid operand to +: 'void'"):
        value(C.binary("+", C.cast(keywords("void"), 1), 1))
    with raises(NotEvaluated, match="the root is not a closed constructor expression: its value has no closed constructor expression"):
        value(C.cast(keywords("void"), 1))
    assert repr(CV.VOID) == "void"

<a id="CCS-03"></a>
## CCS-03 · The conditional: the chosen operand in the two operands' type, C's promoted, C++'s one where they are one; GNU's `a ?: b`

In [ ]:
with store.transaction():
    # The chosen operand, converted to the type the usual arithmetic conversions give the two (ISO/IEC 9899, 6.5.15)
    assert text(C.conditional(1, 2, 3.5)) == "2.0" and text(C.conditional(0, 2, C.literal(3, "u"))) == "3u"
    assert text(C.conditional(1, C.literal(1, "u"), -1)) == "1u" and text(C.conditional(1, 2, C.literal(3, "l"))) == "2l"
    # C promotes two bools to int; C++ keeps the one type both are
    assert text(C.conditional(1, True, False)) == "1" and text(X.conditional(1, True, False), of=X) == "true"
    # The other operand's type is read, not its value; where only its declaration gives it, the chosen one's is kept
    assert text(C.conditional(1, 2, C.binary("/", 1.0, 0.0))) == "2.0" and text(C.conditional(1, 2, x), {"x": 1.5}) == "2.0"
    assert text(C.conditional(1, 2, x)) == "2" and text(C.conditional(1, 2, C.member(x, "a")), {"x": {"a": 1.5}}) == "2"
    assert text(C.conditional(x, 1, 2)) == "x ? 1 : 2" and text(C.conditional(x, 1, 2), {"x": 0}) == "2"
    with raises(TypeError, match="invalid operands to conditional expression ('_Decimal32' and 'double')"):
        value(C.conditional(1, C.literal(0.5, "df"), 1.0))
    # GNU's `a ?: b`: `a`, evaluated once, unless it is zero, then `b`; in their common type
    assert text(gnu(5, 7)) == "5" and text(gnu(0, 7)) == "7" and text(gnu(0, 7.5)) == "7.5" and text(gnu(2, 7.5)) == "2.0"
    assert text(gnu(x, 1)) == "x ?: 1" and text(gnu(x, 1), {"x": 3}) == "3"
    # An operand of no arithmetic type is chosen as it is
    assert text(C.conditional(1, string("a"), string("b"))) == '"a"' and text(C.conditional(1, 2, string("s"))) == "2"
    with raises(TypeError, match="invalid operands to binary + ('_Decimal32' and 'double')"):  # the other's type is still C's
        value(C.conditional(1, 2, C.binary("+", C.literal(0.1, "df"), 1.0)))
    # What C may never evaluate stays, though it would raise: a branch, by an unknown condition; `&&`'s right operand
    assert text(C.conditional(x, 1, C.binary("/", 1, 0))) == "x ? 1 : 1 / 0" and text(C.binary("&&", x, C.binary("/", 1, 0))) == "x && 1 / 0"

<a id="CCS-04"></a>
## CCS-04 · `nullptr`: false, equal to itself and to the literal 0, the operand of nothing else

In [ ]:
with store.transaction():
    # `nullptr`, C23's and C++'s, is a literal; a variable bound to None is one too
    for of in (C, X):
        assert text(nullptr, of=of) == "nullptr" and value(nullptr, of=of) is None and simplified(nullptr, of=of).literal
        assert text(of.identifier("p"), {"p": None}, of=of) == "nullptr"
        # It is false, equal to itself and to the integer literal 0, a null pointer constant
        for spec, written in [(of.binary("==", nullptr, nullptr), "true"), (of.binary("!=", nullptr, 0), "false"),
                              (of.binary("==", paren(0, of), nullptr), "true"), (of.unary("!", nullptr), "true"),
                              (of.conditional(nullptr, 1, 2), "2"), (of.binary("||", nullptr, 0), "false"),
                              (of.cast(keywords("bool"), nullptr), "false")]:
            assert text(spec, of=of) == written, (written, text(spec, of=of))
        # It is the operand of nothing else
        for spec, message in [(of.binary("+", nullptr, 1), "invalid operand to +: 'nullptr_t'"),
                              (of.binary("<", nullptr, nullptr), "invalid operands to binary < ('nullptr_t' and 'nullptr_t')"),
                              (of.binary("==", nullptr, 1), "invalid operands to binary == ('nullptr_t' and 'int')"),
                              (of.cast(keywords("int"), nullptr), "invalid operand to a conversion: 'nullptr_t'")]:
            with raises(TypeError, match=message):
                value(spec, of=of)
    with raises(ValueError, match="a constant needs the singleton nullptr"):  # a malformed term
        value(lambda u: u.constant(lambda r: r.singleton("nil")))

<a id="CCS-05"></a>
## CCS-05 · `sizeof` and `alignof`: a type's layout under its holder's ABI, an operand's type unevaluated, arrays and strings; what stays, by class

In [ ]:
with store.transaction():
    # A type's size and alignment are its holder's ABI's: a `size_t`, written as the `unsigned long` it is under both
    for type_, size in [(keywords("char"), "1ul"), (keywords("_Bool"), "1ul"), (keywords("short"), "2ul"), (keywords("int"), "4ul"),
                        (keywords("long", "long"), "8ul"), (keywords("double"), "8ul"), (named("wchar_t"), "4ul"),
                        (named("_Float16"), "2ul"), (named("_Decimal128"), "16ul")]:
        assert text(sizeof(type_)) == size and text(sizeof(type_), abi=aapcs64()) == size, size
    assert text(sizeof(keywords("long", "double"))) == "16ul" and text(alignof(keywords("long", "double"))) == "16ul"  # x87's 80 bits in 16
    assert rule(sizeof(keywords("long", "double")), abi=aapcs64()).value == 16  # binary128
    # An array's size is its elements', its alignment theirs
    assert text(sizeof(keywords("int", declarator=array(3)))) == "12ul" and text(sizeof(keywords("int", declarator=array(3, array(2))))) == "24ul"
    assert text(alignof(keywords("double", declarator=array(4)))) == "8ul" and text(alignof(keywords("char", declarator=array())))  == "1ul"
    # A string literal is an array of its encoding's code units and a terminating zero
    assert [text(sizeof(s)) for s in (string("abc"), string("é"), string("é", "u8"), string("ab", "u"), string("ab", "U"),
                                      string("ab", "L"), concat("ab", "cd"))] == ["4ul", "3ul", "3ul", "6ul", "12ul", "12ul", "5ul"]
    # An operand is not evaluated; its type is its kind's, C's comparisons an int, C++'s a bool
    assert text(sizeof(C.binary("/", 1, 0))) == "4ul" and text(sizeof(C.binary("+", C.literal(1, "l"), 1.5))) == "8ul"
    assert text(sizeof(C.binary("<", 1, 2))) == "4ul" and text(sizeof(X.binary("<", 1, 2), X), of=X) == "1ul"
    assert text(sizeof(C.cast(keywords("char"), 1))) == "1ul" and text(sizeof(C.unary("-", C.cast(keywords("char"), 1)))) == "4ul"  # promoted
    assert text(sizeof(C.conditional(x, 1, 2.5))) == "8ul" and text(sizeof(C.binary(",", 1, C.literal(1, "u")))) == "4ul"
    # A variable's type is its value's; one the scope does not bind stays
    assert text(sizeof(x), {"x": 2 ** 40}) == "8ul" and text(sizeof(x)) == "sizeof x"
    # What stays, by class: an operand whose declaration gives its type (c), a type that is not arithmetic (d), a layout
    # no ABI dialect stores (a)
    for spec, reason in [(sizeof(C.member(x, "a")), ("C sizeofexpression of an operand of unknown type is not simplified", "c")),
                         (alignof(C.call("f")), ("C alignofexpression of an operand of unknown type is not simplified", "c")),
                         (sizeof(keywords("int", declarator=pointer)), ("C sizeofexpression of a type that is not arithmetic is not simplified", "d")),
                         (sizeof(named("my_t")), ("C sizeofexpression of a type that is not arithmetic is not simplified", "d")),
                         (sizeof(C.literal(5, "wb")), ("C sizeofexpression of a bit-precise integer type is not simplified", "a")),
                         (sizeof(nullptr), ("C sizeofexpression of nullptr_t is not simplified", "a"))]:
        assert reasons(spec, {"x": {"a": 1}})[0] == reason, reason
    # What C makes an error raises
    for spec, error, message in [(sizeof(keywords("void")), TypeError, "invalid application of 'sizeof' to a void type"),
                                 (alignof(keywords("void")), TypeError, "invalid application of '_Alignof' to a void type"),
                                 (sizeof(keywords("int", declarator=array())), TypeError, "invalid application of 'sizeof' to an incomplete type"),
                                 (sizeof(keywords("int", declarator=array(0))), TypeError, "size of an array is not positive: 0"),
                                 (sizeof(keywords("int", declarator=array(1.5))), TypeError, "size of an array has non-integer type 'double'")]:
        with raises(error, match=message):
            value(spec)
    # Every kind's type: parentheses, `sizeof`'s own, `_Generic`'s choice, `!`, a shift
    assert [text(sizeof(spec)) for spec in (paren(1), sizeof(keywords("int")), generic(1, (keywords("int"), 1.5)), C.unary("!", 1),
                                            C.binary("<<", C.literal(1, "l"), 1))] == ["4ul", "8ul", "8ul", "4ul", "8ul"]
    assert text(sizeof(X.unary("!", 1), X), of=X) == "1ul"
    # A conditional's: its operands' common type, `nullptr_t` of two `nullptr`s; an operand whose type is not known stays
    assert reasons(sizeof(C.conditional(x, nullptr, nullptr)))[0] == ("C sizeofexpression of nullptr_t is not simplified", "a")
    for spec in (x, concat("a", x), C.unary("*", x), C.binary(",", 0, string("abc")), C.cast(keywords("char", declarator=pointer), 0),
                 C.conditional(x, 1, nullptr)):
        assert reasons(sizeof(spec), {"x": "abc"} if spec is x else None)[0] == ("C sizeofexpression of an operand of unknown type is not simplified", "c")
    assert reasons(sizeof(X.binary("<=>", 1, 2), X), of=X)[0] == ("Cpp sizeofexpression of an operand of unknown type is not simplified", "c")
    for spec, message in [(C.unary("-", nullptr), "invalid operand to -: 'nullptr_t'"), (C.unary("-", string("ab")), "invalid operand to -: 'array'"),
                          (C.unary("-", C.cast(keywords("void"), 0)), "invalid operand to -: 'void'"),
                          (C.binary("+", C.literal(0.1, "df"), 1.0), "invalid operands to binary + ('_Decimal32' and 'double')"),
                          (C.conditional(x, C.cast(keywords("void"), 0), C.cast(keywords("void"), 1)), "invalid application of 'sizeof' to a void type")]:
        with raises(TypeError, match=message):
            value(sizeof(spec))
    assert reasons(sizeof(lambda u: u.error(lambda r: r.text("1 +"))))[0] == ("C error is not simplified", "b")  # invalid syntax
    assert [text(sizeof(spec)) for spec in (concat("a", string("b", "L")), gnu(1, 1.5), compound(keywords("int"), 5))] == ["12ul", "8ul", "4ul"]
    # An operand `sizeof` does not evaluate stays where the size does, though it would raise
    assert text(sizeof(C.binary("+", C.member(x, "a"), C.binary("/", 1, 0)))) == "sizeof(x.a + 1 / 0)"

<a id="CCS-06"></a>
## CCS-06 · `_Generic`: the association of its controlling operand's type, by C's identity of types, unevaluated; `default`; what stays, by class

In [ ]:
with store.transaction():
    # The association whose type is its controlling operand's, by C's identity of types, else `default`
    int_, long_, long_long = keywords("int"), keywords("long"), keywords("long", "long")
    assert text(generic(1, (long_, 1), (int_, 2), (None, 3))) == "2" and text(generic(1.5, (keywords("float"), 1), (None, 9))) == "9"
    assert text(generic(C.literal(1, "l"), (long_long, 1), (long_, 2))) == "2"  # long and long long are types of their own
    assert text(generic(C.cast(named("int64_t"), 1), (long_long, 1), (long_, 2))) == "2"  # int64_t is long, under both ABIs
    assert text(generic(C.cast(named("int64_t"), 1), (long_long, 1), (long_, 2)), abi=aapcs64()) == "2"
    assert text(generic(C.cast(keywords("char"), 1), (keywords("signed", "char"), 1), (keywords("unsigned", "char"), 2),
                        (keywords("char"), 3))) == "3"  # char is neither signed char nor unsigned char
    assert text(generic(1, (keywords("const", "int"), 1), (None, 2))) == "2"  # a qualified type, which no operand's is
    assert text(generic(1, (keywords("int", declarator=pointer), 1), (int_, 2))) == "2"  # a pointer, which no operand's is
    assert text(generic(C.binary("/", 1, 0), (int_, 7))) == "7"  # its operand is not evaluated
    assert text(generic(keywords("double"), (keywords("double"), 1), (None, 2))) == "1"  # a type name as its operand (C2y)
    assert text(generic(x, (int_, C.binary("+", x, 1)), (keywords("double"), 0)), {"x": 41}) == "42"  # by its value's type
    assert text(generic(x, (int_, C.binary("+", x, 1)))) == "_Generic(x, int: x + 1)"
    # What stays, by class: an operand whose declaration gives its type (c), an association of a type a declaration
    # names (d), which may be the operand's
    assert reasons(generic(C.member(x, "a"), (int_, 1)), {"x": {"a": 1}})[0] == ("C genericselection of an operand of unknown type is not simplified", "c")
    assert reasons(generic(1, (named("my_t"), 1), (int_, 2)))[0] == ("C genericselection of an association of a type a declaration names is not simplified", "d")
    # What C makes an error raises
    for spec, message in [(generic(1.5, (int_, 1)), "'_Generic' selector of type 'double' is not compatible with any association"),
                          (generic(1, (int_, 1), (named("int32_t"), 2)), "'_Generic' specifies two associations of type 'int'")]:
        with raises(TypeError, match=message):
            value(spec)
    # `nullptr` and `void` are types of their own; a type name that is not arithmetic, a string (a pointer once converted)
    assert text(generic(nullptr, (int_, 1), (None, 2))) == "2" and text(generic(C.cast(keywords("void"), 0), (None, 3))) == "3"
    assert reasons(generic(keywords("int", declarator=pointer), (None, 3)))[0] == ("C genericselection of a type that is not arithmetic is not simplified", "d")
    assert reasons(generic(string("abc"), (None, 3)))[0] == ("C genericselection of an operand of unknown type is not simplified", "c")

<a id="CCS-07"></a>
## CCS-07 · Compound literals: a scalar's value, an array's elements by position and designator, braces elided or written, an array of arrays, a character array's string; an array written as a compound literal

In [ ]:
with store.transaction():
    # A compound literal of an arithmetic type is its initializer's value, converted; an empty one its zero
    assert text(compound(keywords("int"), 7)) == "7" and text(compound(keywords("double"), 7)) == "7.0" and text(compound(keywords("int"))) == "0"
    # Of an array, an array, written as a compound literal of its elements' literals; a subscript reads its elements, in
    # its element type
    ints = keywords("int", declarator=array())
    assert text(compound(ints, 10, C.binary("+", 1, 2))) == "(int[]){10, 3}" and text(C.subscript(compound(ints, 10, 20, 30), 1)) == "20"
    assert value(compound(ints, 10, C.binary("+", 1, 2))) == [10, 3] and simplified(compound(ints, 10)).constructed
    assert text(compound(keywords("short", declarator=array(3)), 1, designated(2, 9))) == "(short[]){1, 0, 9}"
    assert text(compound(keywords("unsigned", declarator=array()), 1)) == "(unsigned int[]){1u}"
    assert text(C.subscript(compound(keywords("double", declarator=array()), 1), 0)) == "1.0"
    # Its length is its type's, else its initializer's; an element no item initializes is zero; a designator places one
    assert text(C.subscript(compound(keywords("int", declarator=array(4)), designated(2, 9)), 2)) == "9"
    assert text(C.subscript(compound(keywords("int", declarator=array(4)), designated(2, 9)), 0)) == "0"
    assert text(C.subscript(compound(ints, 1, designated(3, 7, last=5)), 4)) == "7" and text(sizeof(compound(ints, 1, designated(3, 7, last=5)))) == "24ul"
    assert text(sizeof(compound(ints, 1, 2, 3))) == "12ul" and text(sizeof(compound(keywords("short", declarator=array(5))))) == "10ul"
    # A character array's string: its code units and its terminating zero
    chars = keywords("char", declarator=array())
    assert text(C.cast(keywords("int"), C.subscript(compound(chars, "hi"), 1))) == "105" and text(sizeof(compound(chars, "hi"))) == "3ul"
    assert text(C.cast(keywords("int"), C.subscript(compound(keywords("char", declarator=array(2)), "hi"), 1))) == "105"  # no room for its zero
    # What C makes an error raises
    for spec, message in [(compound(keywords("int", declarator=array(2)), 1, 2, 3), "excess elements in an array initializer"),
                          (compound(keywords("int"), 1, 2), "excess elements in a scalar initializer"),
                          (compound(keywords("char", declarator=array(1)), "hi"), "initializer-string for an array is too long"),
                          (compound(ints, designated(-1, 1)), "array index in an initializer is negative: -1")]:
        with raises(TypeError, match=message):
            value(spec)
    # What stays, by class: a structure's members, which a declaration gives (d)
    point = lambda u: u.typeid(lambda i: i.specifiers([lambda s: s.classspecifier(lambda c: c.key("struct").name(  # noqa: E731
        lambda n: n.identifier(lambda d: d.spelling("point"))).body(True))]))
    assert reasons(compound(point, 1, 2))[0] == ("C compoundliteralexpression of a type that is not arithmetic is not simplified", "d")
    assert reasons(compound(keywords("int"), compound(keywords("int"), 1)))  == []  # a compound literal is an item's value
    # A character array of one integer is initialized as any array; a designator is an index, its value not braced
    assert text(C.cast(keywords("int"), C.subscript(compound(chars, 65), 0))) == "65"
    field = lambda u: u.designatedinitializer(lambda r: r.designators([lambda d: d.fielddesignator(  # noqa: E731
        lambda f: f.name(lambda i: i.spelling("a")))]).initializer(lambda i: i.equalinitializer(lambda e: e.value(C.literal(1)))))
    braced = lambda u: u.designatedinitializer(lambda r: r.designators([lambda d: d.indexdesignator(  # noqa: E731
        lambda i: i.index(C.literal(0)))]).initializer(lambda i: i.initializerlist(lambda l: l.items([C.literal(1)]))))
    for spec, message in [(compound(ints, field), "an array's element is designated by its index"),
                          (compound(ints, designated(1.5, 1)), "array index in an initializer has non-integer type 'double'"),
                          (concat("a", x), "invalid operand to a string concatenation: 'int'")]:
        with raises(TypeError, match=message):
            value(spec, {"x": 1})
    assert text(compound(ints, braced)) == "(int[]){1}"  # a scalar's braces
    assert text(sizeof(compound(keywords("int", declarator=array(4))))) == "16ul" and text(C.subscript(compound(keywords("int", declarator=array(4))), 3)) == "0"
    # A call is given an array's elements' natives
    total = CV.Scope({"x": [1, 2]}, functions={"f": lambda items: sum(items)})
    assert value(C.call("f", compound(ints, 1, 2)), total) == 3 and value(C.call("f", x), total) == 3 and value(C.call("f", x), CV.Scope({"x": []}, functions={"f": sum})) == 0

<a id="CCS-08"></a>
## CCS-08 · `_BitInt(N)`: literals of their narrowest type, arithmetic unpromoted and ranked below a standard type, casts, a value written back as its narrowest literal where that is of its type, else as a cast of a literal

In [ ]:
with store.transaction():
    # A `wb` literal is of the narrowest bit-precise integer type that holds it, signed (`_BitInt(N)`, N >= 2) or not
    assert rule(C.literal(5, "wb")).ctype.name() == "_BitInt(4)" and rule(C.literal(5, "uwb")).ctype.name() == "unsigned _BitInt(3)"
    # Its arithmetic is unpromoted, in the operands' common type: 3wb + 4wb, a _BitInt(4) 7, whose narrowest literal 7wb is
    assert text(C.binary("+", C.literal(3, "wb"), C.literal(4, "wb"))) == "7wb"
    # A value whose narrowest literal is of another type is a cast of a literal: 7wb - 6wb is a _BitInt(4) 1, where 1wb is
    # a _BitInt(2); a value no standard type holds, a cast of its narrowest literal
    assert text(C.binary("-", C.literal(7, "wb"), C.literal(6, "wb"))) == "(_BitInt(4))1"
    assert text(C.binary("<<", C.cast(bits(70), 1), 65)) == "(_BitInt(70))36893488147419103232wb"
    assert (rule(C.binary("-", C.literal(7, "wb"), C.literal(6, "wb"))).ctype.name(), rule(C.binary("-", C.literal(7, "wb"), C.literal(6, "wb"))).value) == ("_BitInt(4)", 1)
    # It ranks below a standard type of its width, so an int operand makes it an int; unsigned arithmetic wraps
    assert text(C.binary("+", C.literal(1, "wb"), 1)) == "2" and text(C.binary("+", C.literal(7, "uwb"), C.literal(1, "uwb"))) == "(unsigned _BitInt(3))0"
    assert rule(C.binary("+", C.literal(7, "uwb"), C.literal(1, "uwb"))).value == 0
    with raises(OverflowError, match="signed overflow in +: 2 does not fit '_BitInt(2)'"):
        value(C.binary("+", C.literal(1, "wb"), C.literal(1, "wb")))
    # A cast converts to it, as to any integer type, alike under both ABIs
    assert text(C.cast(bits(7), 200)) == "-56wb" and text(C.cast(bits(7), 200), abi=aapcs64()) == "-56wb"
    assert text(C.cast(bits(70, unsigned=True), -1)) == f"{2 ** 70 - 1}uwb" and text(C.cast(bits(8), 1)) == "(_BitInt(8))1"
    with raises(TypeError, match="width of '_BitInt' is too small: 1"):
        value(C.cast(bits(1), 0))
    with raises(TypeError, match="width of 'unsigned _BitInt' has non-integer type 'double'"):
        value(C.cast(lambda u: u.typeid(lambda i: i.specifiers([lambda s: s.primitivetypespecifier(lambda p: p.keyword("unsigned")),
                                                                lambda s: s.bitintspecifier(lambda b: b.width(C.literal(1.5)))])), 0))
    # An unsigned bit-precise type of a standard signed type's width ranks below it: their common type is its unsigned one
    assert text(C.binary("+", C.cast(bits(32, unsigned=True), 1), 1)) == "2u"
    with raises(TypeError, match="invalid type specifiers with '_BitInt': long"):
        value(C.cast(lambda u: u.typeid(lambda i: i.specifiers([lambda s: s.primitivetypespecifier(lambda p: p.keyword("long")),
                                                                lambda s: s.bitintspecifier(lambda b: b.width(C.literal(8)))])), 1))

<a id="CCS-09"></a>
## CCS-09 · C++'s names: identifiers, qualified names by their spelling, `this`; user-defined literals by the scope's literal operators, of one literal or adjacent strings

In [ ]:
with store.transaction():
    # An identifier is the scope's variable of its name; a qualified name the variable of its spelling
    assert text(X.binary("+", ident("x"), 1), {"x": 2}, of=X) == "3" and text(qualified("ns", "x"), {"ns::x": 5}, of=X) == "5"
    assert text(qualified("std", "numbers", "pi"), {"std::numbers::pi": 3.14}, of=X) == "3.14" and text(qualified("ns", "x"), of=X) == "ns::x"
    # `this` is the scope's variable `this`, read through `->` as any object is
    assert text(X.member(this_, "a", "->"), {"this": {"a": 3}}, of=X) == "3" and text(this_, of=X) == "this"
    # A name that is neither stays: a declaration's (d)
    operator_plus = lambda u: u.idexpression(lambda r: r.name(lambda n: n.operatorname(lambda o: o.operator("+"))))  # noqa: E731
    assert reasons(operator_plus, of=X) == [("Cpp idexpression of a name that is not an identifier is not simplified", "d")]
    # A user-defined literal calls the literal operator the scope provides, by its name, with its literal's value
    km = XV.Scope({}, functions={'operator""_km': lambda metres: metres * 1000})
    assert text(udl(12, "_km"), km, of=X) == "12000" and text(X.binary("+", udl(1, "_km"), udl(2, "_km")), km, of=X) == "3000"
    assert text(udl(12, "_km"), of=X) == "12_km"  # one the scope does not provide stays
    # Adjacent strings with a suffix, `"ab" "c"_len`: the literal operator called with their concatenation
    strings = lambda *parts: lambda u: u.userdefinedliteral(lambda r: r.concatenation(  # noqa: E731
        lambda c: c.parts([X.DIALECT.spec(part) for part in parts])).suffix("_len"))
    length = XV.Scope({"x": 1}, functions={'operator""_len': len})
    assert text(strings(string("ab"), string("c")), length, of=X) == "3" and text(strings(string("ab"), string("c")), of=X) == '"ab" "c"_len'
    with raises(TypeError, match="invalid operand to a string concatenation: 'int'"):
        value(strings(string("a"), ident("x")), length, of=X)
    # A qualified name whose qualifiers are not identifiers stays too
    in_template = lambda u: u.idexpression(lambda r: r.name(lambda n: n.qualifiedname(lambda q: q.qualifiers([  # noqa: E731: v<int>::x
        lambda m: m.templateid(lambda t: t.name(lambda k: k.identifier(lambda i: i.spelling("v"))).arguments([keywords("int")]))]).name(
        lambda m: m.identifier(lambda i: i.spelling("x"))))))
    assert reasons(in_template, of=X)[0] == ("Cpp idexpression of a name that is not an identifier is not simplified", "d")
    global_x = lambda u: u.idexpression(lambda r: r.name(lambda n: n.qualifiedname(  # noqa: E731: ::x
        lambda q: q.global_scope(True).name(lambda m: m.identifier(lambda i: i.spelling("x"))))))
    assert text(global_x, {"::x": 7}, of=X) == "7"

<a id="CCS-10"></a>
## CCS-10 · C++'s casts: functional casts, braced ones refusing a narrowing conversion, `static_cast`; the object model's casts stay

In [ ]:
with store.transaction():
    # A functional cast to an arithmetic type converts as a cast does; with no value, its zero
    assert text(functional("int", 2.9), of=X) == "2" and text(functional("int"), of=X) == "0" and text(functional("double", 2), of=X) == "2.0"
    assert text(functional("unsigned", -1), of=X) == "4294967295u" and text(functional("int", 2, braced=True), of=X) == "2"
    assert text(functional("double", 2, braced=True), of=X) == "2.0" and text(functional("int", braced=True), of=X) == "0"
    # A braced one refuses a narrowing conversion: to bool, from a floating type to an integer one, or one that changes its value
    for spec, message in [(functional("int", 2.5, braced=True), "narrowing conversion of 2.5 from 'double' to 'int'"),
                          (functional("uint8_t", 300, braced=True), "narrowing conversion of 300 from 'int' to 'uint8_t'"),
                          (functional("bool", 1, braced=True), "narrowing conversion of 1 from 'int' to 'bool'"),
                          (functional("int", 1, 2), "a functional cast to 'int' of 2 values")]:
        with raises(TypeError, match=message):
            value(spec, of=X)
    assert text(functional("uint8_t", 200, braced=True), of=X) == "(unsigned char)200"  # no literal: a cast of one, as C writes it
    # `static_cast` is a cast; to `void`, its operand evaluated to no value
    assert text(static_cast(keywords("double"), 1), of=X) == "1.0" and text(static_cast(keywords("int"), 2.9), of=X) == "2"
    assert text(static_cast(keywords("void"), X.binary("+", 1, 2)), of=X) == "static_cast<void>(3)"
    # What stays, by class: the object model's casts, and a cast to a class (d)
    assert reasons(static_cast(keywords("int"), 1, "reinterpret_cast"), of=X)[0] == ("Cpp namedcastexpression 'reinterpret_cast' is not simplified", "d")
    assert reasons(functional("point", 1), of=X)[0] == ("Cpp functionalcastexpression to a type that is not arithmetic is not simplified", "d")
    assert reasons(static_cast(named("point"), 1), of=X)[0] == ("Cpp namedcastexpression to a type that is not arithmetic is not simplified", "d")
    # `void()` is no value; a type a `typename` specifier names is a declaration's
    assert text(functional("void"), of=X) == "void()" and rule(functional("void"), of=X).value is CV.VOID
    typename_t = lambda u: u.functionalcastexpression(lambda r: r.type(lambda s: s.typenamespecifier(  # noqa: E731
        lambda n: n.name(lambda m: m.identifier(lambda i: i.spelling("T"))))).initializer(lambda i: i.parenthesizedinitializer(
        lambda p: p.arguments([X.literal(1)]))))
    assert reasons(typename_t, of=X)[0] == ("Cpp functionalcastexpression to a type that is not arithmetic is not simplified", "d")
    # Each cast's type is its operand's type, unevaluated; a cast to a class's, or by the object model's casts, is not known
    assert text(sizeof(functional("short", 2), X), of=X) == "2ul" and text(sizeof(static_cast(keywords("char"), 1), X), of=X) == "1ul"
    for spec in (static_cast(keywords("int"), 1, "reinterpret_cast"), functional("point", 1)):
        assert reasons(sizeof(spec, X), of=X)[0] == ("Cpp sizeofexpression of an operand of unknown type is not simplified", "c")

<a id="CCS-11"></a>
## CCS-11 · C++'s `<=>`: `<compare>`'s orderings, compared with the literal 0 and with one another, a narrowing conversion refused

In [ ]:
with store.transaction():
    # `<=>` of two integers is a strong ordering; of floating values a partial one, unordered where one is NaN
    three = lambda a, b: X.binary("<=>", a, b)  # noqa: E731
    assert [str(rule(three(*pair), of=X).value) for pair in [(1, 2), (2, 2), (3, 2), (1.0, 2), (2.0, 2), (1.0, math.nan)]] == [
        "std::strong_ordering::less", "std::strong_ordering::equal", "std::strong_ordering::greater",
        "std::partial_ordering::less", "std::partial_ordering::equivalent", "std::partial_ordering::unordered"]
    assert rule(three(1, 2), of=X).value == XV.Ordering("strong_ordering", -1)
    assert text(three(1, 2), of=X) == "1 <=> 2"  # an ordering has no literal
    # It compares with the literal 0, on either side, and with an ordering of its category by == and !=
    assert [text(X.binary(op, three(1, 2), 0), of=X) for op in ("<", "<=", ">", ">=", "==", "!=")] == [
        "true", "true", "false", "false", "false", "true"]
    assert text(X.binary(">", 0, three(1, 2)), of=X) == "true" and str(rule(three(0, three(1, 2)), of=X).value) == "std::strong_ordering::greater"
    assert [text(X.binary(op, three(1.0, math.nan), 0), of=X) for op in ("<", "==", "!=")] == ["false", "false", "true"]
    assert text(X.binary("==", three(2, 2), three(3, 3)), of=X) == "true" and text(X.binary("!=", three(1, 2), three(2, 1)), of=X) == "true"
    # What C++ makes an error raises: a narrowing conversion of an integer, an ordering compared with anything but 0
    for spec, message in [(three(-1, X.literal(0, "u")), "invalid operands to binary <=>: narrowing conversion of -1 from 'int' to 'unsigned int'"),
                          (X.binary("<", three(1, 2), 1), "invalid operands to binary < ('std::strong_ordering' and 'int')"),
                          (X.binary("==", three(1, 2), three(1.0, 2)), "invalid operands to binary == ('std::strong_ordering' and 'std::partial_ordering')")]:
        with raises(TypeError, match=message):
            value(spec, of=X)
    assert str(rule(three(0, three(1.0, math.nan)), of=X).value) == "std::partial_ordering::unordered"
    assert str(rule(three(three(1, 2), 0), of=X).value) == "std::strong_ordering::less"  # an ordering <=> 0 is itself

<a id="CCS-12"></a>
## CCS-12 · C++'s packs: `sizeof...`, pack expansions in a call's arguments, unary and binary folds, empty folds; what stays

In [ ]:
with store.transaction():
    # A pack is the scope's: `sizeof...` its size, an expansion its values in a call's arguments, each by its pattern
    args, k = ident("args"), ident("k")
    scope = XV.Scope({"k": 10}, functions={"f": lambda *values: sum(values)}, packs={"args": [1, 2, 3], "none": []})
    assert text(sizeof_pack("args"), scope, of=X) == "3ul" and text(sizeof_pack("rest"), scope, of=X) == "sizeof...(rest)"
    assert text(X.call("f", expansion(args)), scope, of=X) == "6" and text(X.call("f", expansion(X.binary("*", args, k))), scope, of=X) == "60"
    assert text(X.call("f", 100, expansion(args)), scope, of=X) == "106"
    # A fold: right, `(args op ...)`, `(args op ... op i)`; left, `(... op args)`, `(i op ... op args)`
    assert [text(spec, scope, of=X) for spec in (fold(args, "+", None), fold(None, "-", args), fold(args, "-", 10),
                                                 fold(10, "-", args), fold(args, "*", None))] == ["6", "-4", "-8", "4", "6"]
    # An empty fold: `&&` true, `||` false, `,` no value
    assert text(fold(ident("none"), "&&", None), scope, of=X) == "true" and text(fold(None, "||", ident("none")), scope, of=X) == "false"
    assert rule(fold(ident("none"), ",", None), scope, of=X).value is CV.VOID
    # A pack the scope does not provide stays, named among the free names
    assert text(fold(ident("rest"), "+", None), scope, of=X) == "(rest + ...)" and simplified(fold(ident("rest"), "+", None), scope, of=X).free == {"rest"}
    # What C++ makes an error raises
    two = XV.Scope({}, functions={"f": lambda *values: sum(values)}, packs={"a": [1], "b": [1, 2]})
    for spec, within, message in [(fold(ident("none"), "+", None), scope, "a fold of an empty pack over '+'"),
                                  (X.call("f", expansion(X.binary("+", ident("a"), ident("b")))), two, "the packs a pattern expands are of different lengths"),
                                  (X.call("f", expansion(k)), scope, "a pack expansion's pattern names no parameter pack")]:
        with raises(TypeError, match=message):
            value(spec, within, of=X)
    # A fold over an assignment stays: a write (d)
    assert reasons(fold(args, "=", None), scope, of=X) == [("Cpp foldexpression '=' is not simplified", "d")]
    with raises(TypeError, match="invalid operands to binary == ('nullptr_t' and 'int')"):  # a fold's operands are no literal 0
        value(fold(ident("ps"), "==", None), XV.Scope({}, packs={"ps": [None, 1]}), of=X)

<a id="CCS-13"></a>
## CCS-13 · C++'s `throw`: thrown where it is evaluated, carrying its value; a rethrow stays

In [ ]:
with store.transaction():
    # `throw x` evaluates `x` and throws it, where it is evaluated; a conditional evaluates only the operand it chooses
    c = ident("c")
    assert text(X.conditional(c, 1, throw_(7)), {"c": True}, of=X) == "1"
    with raises(XV.Thrown, match="an exception is thrown"):
        value(X.conditional(c, 1, throw_(7)), {"c": False}, of=X)
    try:
        value(throw_(X.binary("*", 6, 7)), of=X)
        raise AssertionError("not thrown")
    except XV.Thrown as thrown:
        assert thrown.value == 42  # it carries its operand's value
    # A rethrow, `throw` alone, stays: a handler's (d)
    assert reasons(throw_(), of=X) == [("Cpp throwexpression without an operand is not simplified", "d")]
    assert text(X.conditional(c, 1, throw_(7)), of=X) == "c ? 1 : throw 7"  # by an unknown condition, it stays

<a id="CCS-14"></a>
## CCS-14 · What stays in the residual, by class: `FOLDED`, and what each dialect of the family leaves, named

In [ ]:
with store.transaction():
    # The kinds each dialect folds: every kind of class (a), and its literals and names
    assert CV.FOLDED == {"literal", "constant", "concatenatedstring", "variable", "parenthesizedexpression", "genericselection",
                         "subscriptexpression", "callexpression", "memberexpression", "unaryexpression", "sizeofexpression",
                         "alignofexpression", "castexpression", "binaryexpression", "conditionalexpression",
                         "compoundliteralexpression", "extensionexpression"}
    assert XV.FOLDED == CV.FOLDED - {"variable", "genericselection", "compoundliteralexpression"} | {
        "idexpression", "thisexpression", "userdefinedliteral", "functionalcastexpression", "namedcastexpression",
        "sizeofpackexpression", "foldexpression", "packexpansion", "throwexpression"}
    # What stays, named with its class: the object model, writes and statements (d); an initializer alone, a type name (b)
    assign = lambda u: u.assignmentexpression(lambda r: r.left(x).operator("=").right(C.literal(1)))  # noqa: E731
    for spec, reason in [(C.unary("*", x), ("C unaryexpression '*' is not simplified", "d")),
                         (C.unary("&", x), ("C unaryexpression '&' is not simplified", "d")),
                         (assign, ("C assignmentexpression is not simplified", "d")),
                         (lambda u: u.statementexpression(lambda r: r), ("C statementexpression is not simplified", "d")),
                         (lambda u: u.initializerlist(lambda r: r.items([C.literal(1)])), ("C initializerlist is not simplified", "b")),
                         (keywords("int"), ("C Types typeid is not simplified", "b"))]:
        assert reasons(spec, {"x": 1})[0] == reason, reason
    dot_star = X.binary(".*", ident("o"), ident("m"))
    for spec, reason in [(lambda u: u.newexpression(lambda r: r.type(keywords("int"))), ("Cpp newexpression is not simplified", "d")),
                         (lambda u: u.typeidexpression(lambda r: r.operand(X.literal(1))), ("Cpp typeidexpression is not simplified", "c")),
                         (dot_star, ("Cpp binaryexpression '.*' is not simplified", "d")),
                         (static_cast(keywords("int"), 1, "const_cast"), ("Cpp namedcastexpression 'const_cast' is not simplified", "d"))]:
        assert reasons(spec, of=X)[0] == reason, reason
    assert CV.CATEGORIES["C"]["sizeofexpression of an operand of unknown type"] == "c" and XV.CATEGORIES["Cpp"]["foldexpression '+='"] == "d"
    # Inference of a cast to a type that is not arithmetic: not known
    assert C.DIALECT.infer(term(C.cast(keywords("char", declarator=pointer), 0))) is CDialect.Domains.Anything
    enumerator = lambda u: u.enumerator(lambda e: e.name(lambda i: i.spelling("A")))  # noqa: E731: a type name holds none
    assert reasons(C.cast(enumerator, 1))[0] == ("C castexpression to a type that is not arithmetic is not simplified", "d")

<a id="CCS-15"></a>
## CCS-15 · Arrays of arrays and nested braces, initialized as C initializes them, each written as a compound literal; what has no closed constructor expression stays

In [ ]:
with store.transaction():
    # An array of arrays: its items fill its scalars in order, their braces elided, or each subarray its braces' items
    pairs = keywords("int", declarator=array(2, array()))  # int[][2]
    braces = lambda *items: lambda u: u.initializerlist(lambda r: r.items([C.DIALECT.spec(item) for item in items]))  # noqa: E731
    assert text(compound(pairs, 1, 2, 3)) == "(int[][2]){{1, 2}, {3, 0}}" and value(compound(pairs, 1, 2, 3)) == [[1, 2], [3, 0]]
    assert text(compound(pairs, braces(1), 5)) == "(int[][2]){{1, 0}, {5, 0}}" and text(compound(pairs, 1, braces(2), 3)) == "(int[][2]){{1, 2}, {3, 0}}"
    assert text(sizeof(compound(pairs, 1, 2, 3))) == "16ul" and text(C.subscript(C.subscript(compound(pairs, 1, 2, 3), 1), 0)) == "3"
    # A scalar's braces, in an array; `{}` its zero
    assert text(compound(ints, braces(1), 2)) == "(int[]){1, 2}" and text(compound(ints, braces(), 2)) == "(int[]){0, 2}"


    def designation(indices, item):
        """`[i][j] = item`, an index a pair for GNU's range; `item` a list, its braces' items, or an expression."""
        def designator(index):
            first, last = (index, None) if type(index) is int else index
            return lambda d: d.indexdesignator(lambda r: r.index(C.literal(first)) if last is None else r.index(C.literal(first)).last(C.literal(last)))
        initializer = (lambda i: i.initializerlist(lambda l: l.items([C.DIALECT.spec(v) for v in item]))) if type(item) is list else (
            lambda i: i.equalinitializer(lambda e: e.value(C.DIALECT.spec(item))))
        return lambda u: u.designatedinitializer(lambda r: r.designators([designator(index) for index in indices]).initializer(initializer))


    # A designation names an element, a subarray's among them, and the next item follows it; GNU's range repeats one; a
    # later item overrides what an earlier one initialized, braces all of their object
    assert text(compound(pairs, designation([1, 0], 5), 6)) == "(int[][2]){{0, 0}, {5, 6}}"
    assert text(compound(pairs, designation([1], [7, 8]))) == "(int[][2]){{0, 0}, {7, 8}}"
    assert text(compound(keywords("int", declarator=array(2, array(2))), designation([(0, 1)], [1, 2]))) == "(int[][2]){{1, 2}, {1, 2}}"
    assert text(compound(pairs, designation([0, 1], 5), designation([0], [1]))) == "(int[][2]){{1, 0}}"
    # A string initializes the first character array it reaches, in braces or not
    chars3 = keywords("char", declarator=array(3, array()))  # char[][3]
    assert text(compound(chars3, "ab", braces("cd"))) == "(char[][3]){{97, 98, 0}, {99, 100, 0}}"
    assert text(compound(keywords("char", declarator=array(3, array(2, array()))), "ab", "cd")) == "(char[][2][3]){{{97, 98, 0}, {99, 100, 0}}}"
    assert text(compound(chars, C.binary("+", 60, 5))) == "(char[]){65}"  # an expression, which is no string
    # What C makes an error raises
    for spec, message in [(compound(keywords("int", declarator=array(1, array(1))), 1, 2), "excess elements in an array initializer"),
                          (compound(pairs, designation([0, 2], 1)), "excess elements in an array initializer"),
                          (compound(ints, designation([0, 0], 1)), "designator in initializer for scalar type 'int'"),
                          (compound(keywords("int"), designation([0], 1)), "designator in initializer for scalar type 'int'"),
                          (compound(keywords("int"), braces(1)), "too many braces around a scalar initializer"),
                          (compound(ints), "an array of unknown size is initialized by an empty initializer"),
                          (compound(keywords("int", declarator=array(None, array())), 1), "array has incomplete element type")]:
        with raises(TypeError, match=message):
            value(spec)
    # What has no closed constructor expression stays: an array of a value no literal converts to; a list a variable holds,
    # empty or not; C++'s list, which has no compound literal
    third = C.binary("/", C.literal(1.0, "L"), 3)
    assert text(compound(keywords("long", "double", declarator=array()), third)) == "(long double[]){1.0L / 3}"
    assert text(compound(keywords("long", "double", declarator=array(1, array())), braces(third))) == "(long double[][1]){{1.0L / 3}}"
    assert text(x, {"x": [1, 2]}) == "x" and text(x, {"x": []}) == "x" and text(ident("x"), {"x": [1, 2]}, of=X) == "x"

<a id="CCS-16"></a>
## CCS-16 · The scope's option `casts`: `"write"`, the default, writes a value no literal spells as a cast of a literal; `"stay"` leaves it in the residual; any other value is refused

In [ ]:
with store.transaction():
    # The scope's option `casts` (CAP-305): "write", the default, writes a value of a type no literal has as a cast of a
    # literal; "stay" leaves its subterm in the residual, simplified in turn and named by no residue, so that a root that
    # stays is no closed constructor expression
    stay = CV.Scope(options={"casts": "stay"})
    char = C.cast(keywords("char"), 200)
    assert text(char) == text(char, CV.Scope(options={"casts": "write"})) == "(char)-56" and value(char) == -56
    for spec, written in [(char, "(char)200"), (C.cast(keywords("short"), C.binary("+", 1, 2)), "(short)3"),
                          (C.cast(named("uint8_t"), 300), "(uint8_t)300"), (C.cast(bits(4), 1), "(_BitInt(4))1"),
                          (C.binary("+", x, char), "x + (char)200")]:
        kept = simplified(spec, stay)
        assert text(spec, stay) == written and not kept.constructed and kept.residues == (), written
    with raises(NotEvaluated, match="the root is not a closed constructor expression: its value has no closed constructor expression"):
        value(char, stay)
    # It changes only what a cast would write: a value with a literal folds as before, a cast's promoted arithmetic among
    # them, and C's compound literal of an array, whose elements are literals, is written
    for spec in [C.binary("+", char, 0), C.cast(keywords("int"), 2.5), C.binary("*", C.literal(1.5, "f"), 2), compound(chars, "hi")]:
        assert text(spec, stay) == text(spec) and simplified(spec, stay).constructed, text(spec)
    # C++'s scope takes it too; a binding keeps it; the framework's option `errors` goes with it
    assert text(X.cast(keywords("char"), 200), XV.Scope(options={"casts": "stay"}), of=X) == "(char)200"
    assert stay.bind("y", 1).option("casts", "write") == "stay"
    both = CV.Scope(options={"casts": "stay", "errors": "stay"})
    assert text(C.binary("+", char, C.binary("/", 1, 0)), both) == "(char)200 + 1 / 0"
    assert reasons(C.binary("+", char, C.binary("/", 1, 0)), both) == [("C binaryexpression raising ZeroDivisionError is not simplified", "b")]
    # Any other value is refused, before anything is simplified
    with raises(ValueError, match="the option 'casts' is 'write' or 'stay', not 'keep'"):
        simplified(C.binary("+", 1, 2), CV.Scope(options={"casts": "keep"}))

<a id="CCS-17"></a>
## CCS-17 · A parameter assigned a value folds by C's rules, typed under the holder's ABI; one neither assigned nor bound stays where it stood, named, and C's short circuits fold it away

In [ ]:
with store.transaction():
    n = store.References.OfParameter(lambda p: p.name("n"))
    assigned = lambda value: store.Contexts.OfContext(lambda c: c.parameters({  # noqa: E731
        "n": lambda p: p.name("n").value(lambda v: v.value(value))}))
    # Assigned a value, a parameter folds by C's rules, its value typed under the holder's ABI as a variable's is
    assert text(C.binary("+", n, 1), assigned(2)) == "3" and text(C.binary("*", n, 2), assigned(2 ** 40)) == "2199023255552l"
    assert value(C.binary("+", C.identifier("n"), n), CV.Scope({"n": 10}).within(assigned(1))) == 11
    # Neither assigned nor bound, it stays where it stood, named, while its sibling folds around it
    kept = simplified(C.binary("+", n, C.binary("*", 2, 3)))
    assert kept.parameters == {"n"} and F.same(term(kept.residual), term(C.binary("+", n, 6)))
    # An ancestor folds it away where C's short circuit never evaluates it
    assert text(C.binary("&&", 0, n)) == "false" and text(C.binary("||", 1, n)) == "true"
    assert simplified(C.binary("&&", 1, n)).parameters == {"n"}
    # Assigned a term, a parameter is read in its place where the place holds the term's kind, and folds; a term of a
    # kind its place does not hold, C++'s, stays, named
    assert text(C.binary("*", n, 2), assigned(term(C.binary("+", 1, 2)))) == "6"
    assert reasons(C.binary("*", n, 2), assigned(term(X.binary("+", 1, 2), X))) == [
        ("C parameter 'n' assigned a term its place does not hold is not simplified", "b")]
    from mbse.Expressions.C.Types import Expressions as CT
    typeid = F.term(store.Expressions.Framework.Expression(lambda e: e.root(CT.DIALECT.spec(keywords("int")))))
    assert F.holds(store.registered("Expressions.C.Expressions.Term"), typeid)  # C's Term lists C Types' typeid

<a id="CCS-18"></a>
## CCS-18 · A type parameter assigned a type is read in its place: a cast's type and `sizeof`'s operand fold by it, a residual holds it where nothing folds, one neither assigned nor bound stays where it stood, named, and a term its place does not hold stays, named

In [ ]:
from mbse.Expressions.C.Types import Expressions as CT

with store.transaction():
    T = store.References.OfParameter(lambda p: p.name("T"))
    typed = lambda spec: F.term(store.Expressions.Framework.Expression(lambda e: e.root(CT.DIALECT.spec(spec))))  # noqa: E731
    assigned = lambda held: store.Contexts.OfContext(lambda c: c.parameters({  # noqa: E731
        "T": lambda p: p.name("T").value(lambda v: v.value(held))}))
    # A type parameter assigned a type is read in its place: a cast and `sizeof` fold by the type
    assert text(C.cast(T, 3.75), assigned(typed(keywords("int")))) == "3"
    assert text(C.cast(T, 300), assigned(typed(keywords("unsigned", "char")))) == "(unsigned char)44"
    assert text(sizeof(T), assigned(typed(keywords("double")))) == "8ul"  # an operand that may be a type or an expression
    # Where the type lets nothing fold, the residual holds it in the parameter's place
    assert text(C.binary("+", C.cast(T, x), 1), assigned(typed(keywords("int")))) == "(int)x + 1"
    # Neither assigned nor bound, it stays where it stood, named
    for spec in (C.cast(T, 3.75), sizeof(T)):
        kept = simplified(spec)
        assert kept.parameters == {"T"} and F.same(term(kept.residual), term(spec))
    # A term that its place does not hold, an expression where a type is expected, stays, named
    assert reasons(C.cast(T, 3.75), assigned(term(C.binary("+", 1, 2)))) == [
        ("C parameter 'T' assigned a term its place does not hold is not simplified", "b")]